# Équation de Burgers 1D bruitée : Itô vs Stratonovich

On résout l'équation de Burgers visqueuse 1D avec un bruit multiplicatif :

$$\partial_t u + u\,\partial_x u = \nu\,\partial_{xx} u + \sigma\, u \, dW_t$$

où le terme de bruit est interprété soit au sens **d'Itô**, soit au sens de **Stratonovich** ($\circ\,dW_t$).

Comme le bruit est multiplicatif (proportionnel à $u$), les deux interprétations donnent des dynamiques différentes. On le voit en convertissant l'équation de Stratonovich en équation d'Itô équivalente, qui fait apparaître un terme de dérive supplémentaire :

$$\sigma\, u \circ dW_t \;\equiv\; \tfrac{1}{2}\sigma^2 u\, dt + \sigma\, u\, dW_t \quad \text{(au sens d'Itô)}$$

On simule donc les deux schémas d'Euler-Maruyama avec la **même réalisation du bruit** pour comparer directement les trajectoires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## Paramètres et discrétisation

In [ ]:
# Domaine spatial périodique
L = 2 * np.pi
N = 200
x = np.linspace(0, L, N, endpoint=False)
dx = x[1] - x[0]

# Paramètres physiques
nu = 0.05      # viscosité
sigma = 0.15   # intensité du bruit

# Temps
T = 1.0
dt = 2e-4
n_steps = int(T / dt)

# Condition initiale classique pour Burgers
u0 = -np.sin(x)

## Terme déterministe (advection + diffusion)

Discrétisation par différences finies centrées, avec conditions périodiques.

In [ ]:
def burgers_rhs(u, nu, dx):
    u_xp = np.roll(u, -1)
    u_xm = np.roll(u, 1)

    advection = u * (u_xp - u_xm) / (2 * dx)
    diffusion = (u_xp - 2 * u + u_xm) / dx**2

    return -advection + nu * diffusion

## Schémas d'Euler-Maruyama

Un seul incrément de Wiener scalaire $dW_t$ par pas de temps, appliqué de façon multiplicative à $u(x,t)$ sur tout le domaine.

In [ ]:
def simulate(u0, nu, sigma, dt, n_steps, dx, mode="ito", dW_seq=None):
    u = u0.copy()
    if dW_seq is None:
        dW_seq = np.random.randn(n_steps) * np.sqrt(dt)

    for n in range(n_steps):
        dW = dW_seq[n]
        drift = burgers_rhs(u, nu, dx)

        if mode == "stratonovich":
            # correction d'Itô issue de la conversion Stratonovich -> Itô
            drift = drift + 0.5 * sigma**2 * u

        u = u + drift * dt + sigma * u * dW

    return u

## Simulation : même bruit, deux interprétations

In [ ]:
dW_seq = np.random.randn(n_steps) * np.sqrt(dt)

u_deterministic = simulate(u0, nu, 0.0, dt, n_steps, dx, mode="ito")
u_ito = simulate(u0, nu, sigma, dt, n_steps, dx, mode="ito", dW_seq=dW_seq)
u_strato = simulate(u0, nu, sigma, dt, n_steps, dx, mode="stratonovich", dW_seq=dW_seq)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(x, u0, "k--", label="condition initiale", alpha=0.5)
plt.plot(x, u_deterministic, label="déterministe (sans bruit)")
plt.plot(x, u_ito, label="Itô")
plt.plot(x, u_strato, label="Stratonovich")
plt.xlabel("x")
plt.ylabel("u(x, T)")
plt.title(f"Équation de Burgers 1D à T={T}")
plt.legend()
plt.tight_layout()
plt.show()

On observe que les solutions Itô et Stratonovich divergent bien qu'elles partagent la même réalisation de bruit : la correction de dérive $\tfrac12\sigma^2 u$ dans le schéma de Stratonovich modifie l'amplitude de la solution au cours du temps.